In [ ]:
%pip install -q "pandas==2.2.3" "anndata==0.12.16"

In [ ]:
%pip install -q timm

In [ ]:
import math

import torch

from torch.nn.parameter import Parameter
from torch.nn.modules.module import Module

import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import resnet18
import timm

from huggingface_hub import login

from download.HEST_dataset_download import load_hf_token

login(token=load_hf_token())

import os
import zipfile

from huggingface_hub import snapshot_download
from tqdm import tqdm

## Dataset

In [ ]:
from huggingface_hub import login

from download.HEST_dataset_download import load_hf_token

login(token=load_hf_token())

from huggingface_hub import snapshot_download

def download_hest(patterns, local_dir):
    repo_id = 'MahmoodLab/hest'
    snapshot_download(repo_id=repo_id, allow_patterns=patterns, repo_type="dataset", local_dir=local_dir)

local_dir='hest_data' # hest will be dowloaded to this folder

ids_to_query = ['TENX95', 'TENX96', 'TENX97', 'TENX98', 'TENX99'] # list of ids to query

# list_patterns = [f"*{id}[_.]**" for id in ids_to_query]
# download_hest(list_patterns, local_dir)

list_patterns = []
for sample_id in ids_to_query:
    list_patterns.append(f"patches/{sample_id}.h5")
    list_patterns.append(f"st/{sample_id}.h5ad")
download_hest(list_patterns, local_dir) # see method definition above


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

In [ ]:
import numpy as np
import scipy.sparse as sp
from sklearn.neighbors import NearestNeighbors


def make_adjacency(coords, r):
    """
    coords: [N,2]，与 patch 顺序一致
    r: 距离阈值，单位与 coords 相同

    返回：包含权重为 1 的自连接、未归一化的邻接矩阵 [N,N]
    """
    coords = np.asarray(coords, dtype=np.float64)

    if coords.ndim != 2 or coords.shape[1] != 2 or len(coords) == 0:
        raise ValueError("coords must be [N, 2]")

    if not np.isfinite(coords).all():
        raise ValueError("coords must be limited")

    if not np.isfinite(r) or r <= 0:
        raise ValueError("r must be positive")

    neighbors = NearestNeighbors(radius=r).fit(coords)

    A = neighbors.radius_neighbors_graph(mode="connectivity")

    A = A.maximum(A.T).astype(np.uint8)
    A.setdiag(0)
    A.eliminate_zeros()
    A = A + sp.eye(A.shape[0], dtype=A.dtype, format="csr")

    return A

In [ ]:
from pathlib import Path

import anndata as ad
import h5py
import numpy as np
import scipy.sparse as sp
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader

class HESTDataset(Dataset):
    def __init__(self, root, gene_names, transform, radius=550):
        root = Path(root)
        self.root = root / "patches"
        self.gt = root / "st"
        self.transform = transform
        self.r = radius
        self.gene_names = list(gene_names)
        self.num_genes = len(self.gene_names)
        self.path = sorted(self.root.glob("*.h5"))
        self.gt_path = [self.gt / f"{path.stem}.h5ad" for path in self.path]

    def __len__(self):
        return len(self.path)

    def __getitem__(self, index):
        with h5py.File(self.path[index], "r") as f:
            key = "barcodes" if "barcodes" in f else "barcode"
            barcodes = f[key].asstr()[:].reshape(-1)
            coords = f["coords"][:]

            images = []
            for patch in f["img"]:
                patch = Image.fromarray(patch).convert("RGB")
                images.append(self.transform(patch))
            image = torch.stack(images)  # [N,3,H,W]

        # 按 patch 的 barcode 顺序和指定的基因顺序读取标签
        data = ad.read_h5ad(self.gt_path[index])
        label = data[barcodes, self.gene_names].X
        if sp.issparse(label):
            label = label.toarray()
        label = torch.as_tensor(label, dtype=torch.float32)

        A = make_adjacency(coords, r=self.r).astype(np.float32).tocoo()
        # inv_degree = np.asarray(A.sum(axis=1)).ravel() ** -0.5
        # D_inv = sp.diags(inv_degree)
        # A = (D_inv @ A @ D_inv).tocoo()

        indices = torch.from_numpy(np.stack([A.row, A.col]).astype(np.int64))
        values = torch.from_numpy(A.data)
        A = torch.sparse_coo_tensor(indices, values, A.shape).coalesce()

        return ((image, A), label)


def collate_one_slide(batch):
    if len(batch) != 1:
        raise ValueError("一张切片一个样本，请设置 batch_size=1")
    return batch[0]


def make_dataloader(
    dataset,
    batch_size=1,
    shuffle=False,
    num_workers=0,
    collate_fn=None,
):
    return DataLoader(
        dataset=dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        collate_fn=collate_one_slide if collate_fn is None else collate_fn,
    )



## GCN

In [ ]:
class GraphConvolution(Module):
    """
    Simple GCN layer, similar to https://arxiv.org/abs/1609.02907
    """

    def __init__(self, in_features, out_features, bias=True):
        super(GraphConvolution, self).__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.weight = Parameter(torch.FloatTensor(in_features, out_features))
        if bias:
            self.bias = Parameter(torch.FloatTensor(out_features))
        else:
            self.register_parameter('bias', None)
        self.reset_parameters()

    def reset_parameters(self):
        stdv = 1. / math.sqrt(self.weight.size(1))
        self.weight.data.uniform_(-stdv, stdv)
        if self.bias is not None:
            self.bias.data.uniform_(-stdv, stdv)

    def forward(self, input, adj):
        support = torch.mm(input, self.weight)
        output = torch.spmm(adj, support)
        if self.bias is not None:
            return output + self.bias
        else:
            return output

    def __repr__(self):
        return self.__class__.__name__ + ' (' \
               + str(self.in_features) + ' -> ' \
               + str(self.out_features) + ')'


class GCN(nn.Module):
    def __init__(self, nfeat, nhid, nclass, dropout):
        super(GCN, self).__init__()

        self.gc1 = GraphConvolution(nfeat, nhid)
        self.gc2 = GraphConvolution(nhid, nclass)
        self.dropout = dropout

    def forward(self, x, adj):
        x = F.relu(self.gc1(x, adj))
        x = F.dropout(x, self.dropout, training=self.training)
        x = self.gc2(x, adj)
        return x

## Resnet

In [ ]:
class ResNet(nn.Module):
    def __init__(self, input_dim=256, genes=100, side=32):
        super().__init__()

        self.adapter = nn.Sequential(
            nn.Linear(input_dim, side * side),
            nn.Unflatten(1, (1, side, side)),
        )

        self.backbone = resnet18(
            weights=None,
            norm_layer=lambda channels: nn.GroupNorm(8, channels),
        )

        self.backbone.conv1 = nn.Conv2d(in_channels=1, out_channels=64, kernel_size=7, stride=2, padding=3, bias=False)
        self.backbone.fc = nn.Linear(self.backbone.fc.in_features, genes)
        self.activation = nn.ReLU()

    def forward(self, x):
        # [N,256] → [N,1,32,32]
        x = self.adapter(x)

        # [N,1,32,32] → [N,G]
        x = self.backbone(x)
        x = self.activation(x)

        return x

## UNI

In [ ]:
import os
from huggingface_hub import login, hf_hub_download

from download.HEST_dataset_download import load_hf_token

login(token=load_hf_token())

local_dir = "./ckpts/"
os.makedirs(local_dir, exist_ok=True)  # create directory if it does not exist
hf_hub_download("MahmoodLab/UNI", filename="pytorch_model.bin", local_dir=local_dir, force_download=True)

print("FINISH")

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.21GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

FINISH


In [ ]:
class UNI(nn.Module):
    def __init__(self, weights_path, frozen=True):
        super().__init__()
        self.backbone = timm.create_model(
            "vit_large_patch16_224",
            img_size=224,
            patch_size=16,
            init_values=1e-5,
            num_classes=0,
            dynamic_img_size=True,
        )
        weights = torch.load(weights_path, map_location="cpu", weights_only=True)
        self.backbone.load_state_dict(weights, strict=True)
        self.frozen = frozen
        self.backbone.requires_grad_(not frozen)
        if frozen:
            self.backbone.eval()
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def train(self, mode=True):
        super().train(mode)
        if self.frozen:
            self.backbone.eval()
        return self

    def forward(self, patches):
        if patches.shape[-1] == 3:
            patches = patches.permute(0, 3, 1, 2)
        patches = (patches.float() / 255.0 - self.mean) / self.std
        if self.frozen:
            with torch.no_grad():
                return self.backbone(patches)
        else:
            return self.backbone(patches)

## Model

In [ ]:
class MyModel(nn.Module):
    def __init__(self, feature_dim=1024, genes=100):
        super().__init__()
        self.uni = UNI(weights_path="./ckpts/pytorch_model.bin")
        self.gcn = GCN(nfeat=feature_dim, nhid=256, nclass=256, dropout=False)
        self.resnet = ResNet(input_dim=256, genes=genes)

    def forward(self, x, A_norm):
        print("Input")
        x = self.uni(x)
        print("UNI")
        x = self.gcn(x, A_norm)
        print("GCN")
        x = self.resnet(x)
        print("RESNET")
        return x

## Train

In [ ]:
from pathlib import Path
from torchvision.transforms import PILToTensor
from torch.utils.checkpoint import checkpoint

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

D = 1024
G = 10
epochs = 3
uni_batch_size = 8
resnet_batch_size = 64
root = Path("hest_data")

# 先用各切片共有的前 G 个基因；也可替换为你自己确定的目标基因列表。
patch_paths = sorted((root / "patches").glob("*.h5"))
if not patch_paths:
    raise FileNotFoundError(f"没有找到 patch 文件：{root / 'patches'}")
gene_names = None
for path in patch_paths:
    data = ad.read_h5ad(root / "st" / f"{path.stem}.h5ad", backed="r")
    names = data.var_names.tolist()
    data.file.close()
    gene_names = names if gene_names is None else [g for g in gene_names if g in names]
if len(gene_names) < G:
    raise ValueError(f"共有基因只有 {len(gene_names)} 个，请调小 G")
gene_names = gene_names[:G]

# UNI.forward 已负责 /255 和归一化，这里保留 0~255 的 uint8。
dataset = HESTDataset(
    root=root,
    gene_names=gene_names,
    transform=PILToTensor(),
    radius=550,
)
loader = make_dataloader(dataset, batch_size=1, shuffle=True, num_workers=0)

model = MyModel(feature_dim=D, genes=G).to(device)
optimizer = torch.optim.AdamW(
    (p for p in model.parameters() if p.requires_grad),
    lr=1e-4,
)
loss_history = []
print(f"device={device}, 切片数={len(dataset)}, 目标基因数={G}")


for epoch in range(epochs):
    model.train()
    model.uni.eval()
    total_loss = 0.0

    for (patches, A), Y in tqdm(loader, desc=f"Epoch {epoch + 1}/{epochs}"):
        A = A.to(device)
        Y = Y.to(device)
        optimizer.zero_grad(set_to_none=True)
        Y_pred = model(patches, A)
        assert Y_pred.shape == Y.shape, f"预测 {Y_pred.shape}，标签 {Y.shape}"
        loss = F.mse_loss(Y_pred, Y)
        assert torch.isfinite(loss), "Loss 出现 NaN 或 Inf"

        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    epoch_loss = total_loss / len(loader)
    loss_history.append(epoch_loss)
    print(f"Epoch [{epoch + 1}/{epochs}] Train MSE: {epoch_loss:.6f}")


# for epoch in range(epochs):
#     model.train()
#     model.uni.eval()
#     total_loss = 0.0

#     for (patches, A_norm), Y in tqdm(loader, desc=f"Epoch {epoch + 1}/{epochs}"):
#         A_norm = A_norm.to(device)
#         Y = Y.to(device)
#         optimizer.zero_grad(set_to_none=True)

#         # 与 MyModel.forward 顺序一致；UNI 分小批次计算，原始图像保留在 CPU。
#         features = []
#         print("FEATURE")
#         with torch.no_grad():
#             for batch in patches.split(uni_batch_size):
#                 features.append(model.uni(batch.to(device)))
#         X = torch.cat(features, dim=0)  # [N,1024]
#         del features, batch
#         H = model.gcn(X, A_norm)        # [N,256]
#         print("RESNET")
#         # ResNet 分块，并在反向时重算中间激活，避免保存整切片的卷积激活。
#         predictions = [
#             checkpoint(model.resnet, batch, use_reentrant=False)
#             for batch in H.split(resnet_batch_size)
#         ]
#         Y_pred = torch.cat(predictions, dim=0)  # [N,G]
#         assert Y_pred.shape == Y.shape, f"预测 {Y_pred.shape}，标签 {Y.shape}"
#         loss = F.mse_loss(Y_pred, Y)
#         assert torch.isfinite(loss), "Loss 出现 NaN 或 Inf"

#         loss.backward()
#         optimizer.step()
#         total_loss += loss.item()
#         del patches, A_norm, Y, X, H, predictions, Y_pred, loss

#     epoch_loss = total_loss / len(loader)
#     loss_history.append(epoch_loss)
#     print(f"Epoch [{epoch + 1}/{epochs}] Train MSE: {epoch_loss:.6f}")


In [ ]:
from torchvision.transforms import PILToTensor
from pathlib import Path
from torch.utils.checkpoint import checkpoint



device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

root = Path("hest_data")
patch_paths = sorted((root / "patches").glob("*.h5"))
gene_names = None
for path in patch_paths:
    data = ad.read_h5ad(root / "st" / f"{path.stem}.h5ad", backed="r")
    names = data.var_names.tolist()
    data.file.close()
    gene_names = names if gene_names is None else [g for g in gene_names if g in names]
if len(gene_names) < 100:
    raise ValueError(f"共有基因只有 {len(gene_names)} 个，请调小 G")
gene_names = gene_names[:100]
dataset = HESTDataset(
    root=root,
    gene_names=gene_names,
    transform=PILToTensor(),
    radius=550,
)

N = 16
D = 1024
G = 100

X = torch.randint(
    low=0,
    high=256,
    size=(N, 3, 224, 224),
    dtype=torch.uint8,
)

print("X shape: ", X.shape)

# X = torch.randn(N, D)
A_norm = torch.randn(N, N)
Y = torch.rand(N, G)
print("A_norm shape: ", A_norm.shape)
print("Y shape: ", Y.shape)

loader = make_dataloader(dataset, batch_size=1, shuffle=True, num_workers=0)
for (patches, A_norm_), Y_ in loader:
    print("Patch: ", patches.shape)
    print("A: ", A_norm_.shape)
    print("Y: ", Y_.shape)
    break

model = MyModel(feature_dim=D, genes=G).to(device)

X = X.to(device)
A_norm = A_norm.to(device)
Y = Y.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

model.train()
optimizer.zero_grad(set_to_none=True)

Y_pred = model(X, A_norm)
loss = F.mse_loss(Y_pred, Y)

assert Y_pred.shape == Y.shape
assert torch.isfinite(loss)

loss.backward()
optimizer.step()


X shape:  torch.Size([16, 3, 224, 224])
A_norm shape:  torch.Size([16, 16])
Y shape:  torch.Size([16, 100])


/tmp/ipykernel_1574/357538204.py:52: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:760.)
  A = torch.sparse_coo_tensor(indices, values, A.shape).coalesce()


Patch:  torch.Size([20289, 3, 224, 224])
A:  torch.Size([20289, 20289])
Y:  torch.Size([20289, 100])
Input
UNI
GCN
RESNET
